# 11 — Avaliação de notas e desenho experimental

**DATA SCIENCE KNOWLEDGE BASE**  
**Área:** `09_recommender_systems` — Sistemas de Recomendação  
**Capítulo:** 11 de 16  

> Material didático autoral revisado a partir dos notebooks de estudo. Execute as células em sequência, de cima para baixo, em um kernel reiniciado. Consulte as fontes históricas somente para estudo, preservando sua proveniência.

## Objetivos de aprendizagem
- Estabelecer baseline comparável
- Separar dados de treino, validação e teste

## Materiais complementares

- [Guia Técnico](reference_materials/11_avaliacao_mae_rmse.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_11_avaliacao_mae_rmse.pdf)

**Navegação:** os links são relativos à pasta deste notebook. A versão editorial desses PDFs será revalidada após a consolidação científica dos notebooks.

## Fundamentação conceitual
MAE mede o erro absoluto médio e RMSE penaliza erros grandes. Não ajuste hiperparâmetros no conjunto de teste. Divisões temporais são preferíveis quando o cenário exige prever o futuro.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Partição 60/20/20

In [2]:
from sklearn.model_selection import train_test_split
train,temp=train_test_split(ratings,test_size=.4,random_state=7)
val,test=train_test_split(temp,test_size=.5,random_state=7)
print(len(train),len(val),len(test));print('Cobertura de usuários conhecidos:',test.userId.isin(train.userId).mean())

60501 20167 20168
Cobertura de usuários conhecidos: 1.0


**Interpretação.** Essa partição tem dimensão ilustrativa e não bloqueia vazamento temporal.

## Baselines constantes e por item

In [3]:
from sklearn.metrics import mean_absolute_error,mean_squared_error
mu=train.rating.mean();means=train.groupby('movieId').rating.mean()
def metrics(part):
    pred_const=np.full(len(part),mu)
    pred_item=part.movieId.map(means).fillna(mu).to_numpy()
    return {'MAE_global':mean_absolute_error(part.rating,pred_const),'RMSE_global':np.sqrt(mean_squared_error(part.rating,pred_const)),'MAE_item':mean_absolute_error(part.rating,pred_item),'RMSE_item':np.sqrt(mean_squared_error(part.rating,pred_item))}
print('Validação:',metrics(val));print('Teste (relatório):',metrics(test))

Validação: {'MAE_global': 0.8292901816976231, 'RMSE_global': np.float64(1.0459055226356218), 'MAE_item': 0.7626639660132725, 'RMSE_item': np.float64(0.986666476598795)}
Teste (relatório): {'MAE_global': 0.8265998534903133, 'RMSE_global': np.float64(1.0448464435743248), 'MAE_item': 0.7636777845691473, 'RMSE_item': np.float64(0.9934735352312718)}


**Interpretação.** Treine com `train`, escolha configurações em `val` e reporte em `test`.

## Exemplo de corte temporal

In [4]:
temporal=ratings.sort_values('timestamp'); cutoff=temporal.timestamp.quantile(.8)
train_t=temporal[temporal.timestamp<=cutoff];test_t=temporal[temporal.timestamp>cutoff]
print('Treino temporal:',len(train_t),'Teste temporal:',len(test_t))

Treino temporal: 80669 Teste temporal: 20167


**Interpretação.** O corte temporal protege a ordem global dos eventos, mas muda a população observada.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** divisão temporal. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **divisão temporal** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Laboratório adicional — RMSE não é equivalente a ranking

Experimento independente do estado das células anteriores. Use-o como verificação e explique seus limites; resultados em conjuntos sintéticos não são estimativas de desempenho real.

In [5]:
import numpy as np
real=np.array([5,4,1,2]);pred_a=np.array([4,3,2,1]);pred_b=np.array([5,1,4,2]);
for nome,p in [('A',pred_a),('B',pred_b)]:
 print(nome,'RMSE',round(np.sqrt(np.mean((real-p)**2)),3),'top2',np.argsort(-p)[:2].tolist())

A RMSE 1.0 top2 [0, 1]
B RMSE 2.121 top2 [0, 2]


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **divisão temporal** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Avaliação temporal completa e separação por objetivo

MAE e RMSE avaliam previsão numérica de notas; não garantem melhores listas Top-K. Para simular previsão do futuro, faça o corte por `timestamp` **antes** de calcular médias e ajustar modelos. No conjunto final, nunca utilize o teste para selecionar hiperparâmetros.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [6]:
# Avaliação temporal: ajustar baseline apenas no passado
media_passado = train_t.rating.mean()
item_passado = train_t.groupby('movieId').rating.mean()
pred_temporal = test_t.movieId.map(item_passado).fillna(media_passado).to_numpy()
from sklearn.metrics import mean_absolute_error, mean_squared_error
print('MAE temporal (média por item):', round(float(mean_absolute_error(test_t.rating,pred_temporal)),3))
print('RMSE temporal (média por item):', round(float(np.sqrt(mean_squared_error(test_t.rating,pred_temporal))),3))
print('Itens novos no período de teste:',int((~test_t.movieId.isin(train_t.movieId)).sum()))

MAE temporal (média por item): 0.81
RMSE temporal (média por item): 1.046
Itens novos no período de teste: 3044


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
Uma avaliação séria inclui intervalos de confiança e controle de usuários/itens frios.

## Exercícios
A melhor MAE garante o melhor ranking de 10 filmes?

## Soluções comentadas
Não. Ordenação e erro de notas medem aspectos diferentes e podem produzir resultados conflitantes.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [7]:
# EP_OUTPUTS_09_V1
# Exportação após os experimentos. Executar este notebook desde o início (Run All).
# Arquivos com prefixo do capítulo evitam colisões com outras saídas.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
linhas = []
for nome, part in [("validacao_aleatoria", val), ("teste_aleatorio", test)]:
    linhas.append({"particao": nome, "n_interacoes": len(part), **metrics(part)})
linhas.append({"particao": "teste_temporal", "n_interacoes": len(test_t),
    "MAE_item": float(mean_absolute_error(test_t.rating, pred_temporal)),
    "RMSE_item": float(np.sqrt(mean_squared_error(test_t.rating, pred_temporal)))})
pd.DataFrame(linhas).to_csv(out / "11_metricas_baselines.csv", index=False)
print('Outputs do capítulo 11 salvos em:', out.resolve())


Outputs do capítulo 11 salvos em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
